# Advanced Robotics Project, Week 1
## Notebook 2: a mobile robot in a warehouse

Obuda University, Antal Bejczy Center for Intelligent Robotics

A two-wheeled robot learns to work in a small warehouse. We build it from the bottom up: how the wheels move the robot, how it drives to a point, how it finds a way around the shelves, and how it works through a list of places.

| Part | Topic | You write |
|---|---|---|
| A | how a differential-drive robot moves | two lines of the motion step |
| B | driving to a point | the two errors of the controller |
| C | a warehouse map and the A* planner | nothing |
| D | visiting several places with a state machine | two short rules |
| E | a blocked aisle and a first behaviour tree | nothing |

**Working through it.** Run the cells from top to bottom with Shift+Enter. You only type between two marker lines like these:

```
    # ---- your code: 2 lines ----
    ...
    # ----------------------------
```

The cell after each of them is a check. It prints `Correct:` or `Not yet:` followed by a hint. If the notebook gets into a strange state, use Runtime → Restart session and run all.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import Polygon
from IPython.display import HTML
from ipywidgets import interact, FloatSlider
import heapq

np.set_printoptions(precision=3, suppress=True)
print("Setup OK")

---
# Part A: the differential drive

A differential-drive robot has two driven wheels, one on each side. Equal wheel speeds drive it straight, unequal speeds turn it. Its **state** is where it is and which way it faces, $(x, y, \theta)$, with the heading $\theta$ in radians (`th` in the code). We command two numbers: the forward speed $v$ in m/s and the turning rate $\omega$ in rad/s (`w`).

There is no command for moving sideways, and the wheels could not produce one anyway. Robots with this limitation are called **nonholonomic**, which is why parallel parking takes several moves.

### Exercise A1: one motion step

During one short time step `dt` (0.05 s) the robot rolls forward by $v\,\Delta t$ in the direction it faces and turns by $\omega\,\Delta t$. Facing the angle $\theta$ means that forward is the direction $(\cos\theta, \sin\theta)$, so the forward distance splits into an x part and a y part. The x line is already written; the y line is its twin with sine instead of cosine.

Example: facing $\theta = 90°$ (straight up) and driving 1 m, the x part is $1 \cdot \cos 90° = 0$ and the y part is $1 \cdot \sin 90° = 1$, so the robot ends 1 m higher.

`np.sin` and `np.cos` expect the angle in radians (90° is $\pi/2$). `wrap(a)`, defined in the same cell, folds any angle back into −180°…180°, so 190° becomes −170°.

Your lines: the new `y` is `y + v · sin(th) · dt`, and the new `th` is `th + w · dt` passed through `wrap`.

In [ ]:
def wrap(a):
    """Fold an angle (radians) into -pi ... pi."""
    return (a + np.pi) % (2 * np.pi) - np.pi

def step(state, v, w, dt=0.05):
    """Move the robot (x, y, th) forward by one time step dt, with speed v and turning rate w."""
    x, y, th = state
    x = x + v * np.cos(th) * dt
    # ---- your code: 2 lines ----
    y = y
    th = th
    # ----------------------------
    return np.array([x, y, th])

In [ ]:
# Check
def _check_A1():
    s = step(np.array([0.0, 0.0, np.pi / 2]), v=1.0, w=0.0, dt=1.0)
    if not np.allclose(s[:2], [0.0, 1.0], atol=1e-6):
        return (f"Not yet: facing 90° and driving 1 m should end at (0.00, 1.00), you got ({s[0]:.2f}, {s[1]:.2f}). "
                "The y line needs np.sin(th).")
    s = step(np.array([0.0, 0.0, 0.0]), v=0.0, w=1.0, dt=0.5)
    if not np.isclose(s[2], 0.5):
        return f"Not yet: turning at w = 1 rad/s for 0.5 s should give th = 0.50, you got {s[2]:.2f}. The th line adds w * dt."
    s = step(np.array([0.0, 0.0, 3.1]), v=0.0, w=1.0, dt=0.1)
    if not (-np.pi <= s[2] <= np.pi):
        return f"Not yet: the heading left the -180°...180° range (th = {s[2]:.2f} rad). Put wrap(...) around the new heading."
    return "Correct: the robot moves along its heading and turns, but never sideways."

try:
    print(_check_A1())
except Exception as e:
    print(f"Not yet: the check could not run your code ({type(e).__name__}: {e}). Run the cell above and look for a typo.")

Move the sliders. A constant $v$ and $\omega$ always drive a circle of radius $v/\omega$; $\omega = 0$ gives a straight line and $v = 0$ a turn on the spot. Every path of this robot is pieced together from such arcs.

In [ ]:
def robot_patch(state, size=0.35, color="#D85A30"):
    """A triangle pointing along the heading, for drawing."""
    x, y, th = state
    R = np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]])
    pts = np.array([[size, 0], [-size*0.6, size*0.5], [-size*0.6, -size*0.5]]) @ R.T + [x, y]
    return Polygon(pts, closed=True, color=color)

def show_drive(v=1.0, w=0.5, T=4.0):
    s, tr = np.array([0.0, 0.0, 0.0]), []
    for _ in range(int(T / 0.05)):
        tr.append(s[:2]); s = step(s, v, w)
    tr = np.array(tr)
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot(tr[:, 0], tr[:, 1], color="#888")
    ax.add_patch(robot_patch(s))
    ax.set_aspect("equal"); ax.grid(alpha=.3)
    ax.set_xlim(-4, 4); ax.set_ylim(-4, 4)
    radius = f"{abs(v / w):.2f} m" if w else "infinite (straight line)"
    ax.set_title(f"v = {v:.1f} m/s, w = {w:.1f} rad/s  ->  circle radius {radius}")
    plt.show()

interact(show_drive,
         v=FloatSlider(1.0, min=-1.5, max=1.5, step=0.1),
         w=FloatSlider(0.5, min=-2.0, max=2.0, step=0.1));

---
# Part B: go-to-goal

To reach a point, the controller looks at two errors: the **distance error** $\rho$ (`rho`), how far away the goal is, and the **heading error** $\alpha$ (`alpha`), how far the robot has to turn to face it. Two proportional rules turn them into commands:

$$v = k_\rho\,\rho, \qquad \omega = k_\alpha\,\alpha$$

The further the goal, the faster the robot drives; the more its heading is off, the harder it turns. That is **proportional feedback**, the same idea as the arm controller in Notebook 1. The two rules are already in the code below; you compute the two errors they need.

### Exercise B1: the two errors

`dx` and `dy` are how far the goal is along x and along y. The distance error is the straight-line length of that offset. The heading error is the direction of the goal minus the direction the robot faces, wrapped so that the robot always turns the short way.

Example: the robot stands at (0, 0) facing 0°, the goal is at (0, 1). Then $\rho$ = 1 m, the goal lies at 90°, and $\alpha$ = 90° − 0° = +90°, a left turn. If the robot faces 170° and the goal lies at −170°, the plain difference is −340°; `wrap` turns it into +20°.

`np.hypot(dx, dy)` computes $\sqrt{dx^2 + dy^2}$. `np.arctan2(dy, dx)` is the direction of the vector (dx, dy) in radians, between −π and π. The plain `np.arctan(dy / dx)` would not do: (1, 1) and (−1, −1) give the same ratio but point in opposite directions, and dx = 0 would divide by zero.

Your lines: `rho` is `np.hypot(dx, dy)`, and `alpha` is `wrap` of the goal direction `np.arctan2(dy, dx)` minus `th`.

In [ ]:
def go_to_goal(state, goal, k_rho=0.8, k_alpha=2.5, v_max=1.0):
    """Proportional go-to-goal controller. Returns the command (v, w)."""
    x, y, th = state
    dx, dy = goal[0] - x, goal[1] - y
    # ---- your code: 2 lines ----
    rho = 0.0
    alpha = 0.0
    # ----------------------------
    v = min(k_rho * rho, v_max)
    w = k_alpha * alpha
    if abs(alpha) > np.pi / 2:      # goal behind the robot: turn on the spot first
        v = 0.0
    return v, w

In [ ]:
# Check
def _check_B1():
    deg = lambda w: np.rad2deg(w / 2.5)
    v, w = go_to_goal(np.array([0.0, 0.0, np.arctan2(0.4, 0.3)]), (0.3, 0.4))
    if not np.isclose(v, 0.8 * 0.5):
        return f"Not yet: the goal is 0.5 m straight ahead, so rho should be 0.50, yours is {v / 0.8:.2f}. Use np.hypot(dx, dy)."
    if not np.isclose(w, 0.0, atol=1e-6):
        return f"Not yet: the robot already faces the goal, so alpha should be 0°, yours is {deg(w):.0f}°. Subtract th from the goal direction."
    v, w = go_to_goal(np.array([0.0, 0.0, 0.0]), (0.0, 1.0))
    if np.isclose(w, -2.5 * np.pi / 2):
        return "Not yet: alpha has the wrong sign. It is the goal direction minus th, not th minus the goal direction."
    if not np.isclose(w, 2.5 * np.pi / 2):
        return f"Not yet: a goal 90° to the left should give alpha = +90°, yours is {deg(w):.0f}°. The goal direction is np.arctan2(dy, dx)."
    goal = (np.cos(np.deg2rad(-170.0)), np.sin(np.deg2rad(-170.0)))
    v, w = go_to_goal(np.array([0.0, 0.0, np.deg2rad(170.0)]), goal)
    if not np.isclose(w, 2.5 * np.deg2rad(20.0)):
        return f"Not yet: facing 170° with the goal at -170° the robot should turn +20°, yours is {deg(w):.0f}°. Put wrap(...) around alpha."
    return "Correct: rho and alpha are the two errors, and with the two rules below them this is proportional feedback."

try:
    print(_check_B1())
except Exception as e:
    print(f"Not yet: the check could not run your code ({type(e).__name__}: {e}). Run the cell above and look for a typo.")

Move the goal around, and try a start heading of 180° so that the goal is behind the robot: it turns on the spot before it drives. If you are curious, remove `wrap` from your `alpha` line, run the cells again and watch the robot turn the long way round; a forgotten wrap is the classic bug in navigation code. Put it back afterwards.

In [ ]:
def simulate_to(goal, start=(0, 0, 0), T=8.0, tol=0.1):
    s, tr = np.array(start, float), []
    for _ in range(int(T / 0.05)):
        tr.append(s.copy())
        if np.hypot(goal[0] - s[0], goal[1] - s[1]) < tol:
            break
        v, w = go_to_goal(s, goal)
        s = step(s, v, w)
    return np.array(tr)

def show_goal(gx=3.0, gy=2.0, start_heading_deg=0.0):
    tr = simulate_to((gx, gy), start=(0, 0, np.deg2rad(start_heading_deg)))
    fig, ax = plt.subplots(figsize=(5.5, 5))
    ax.plot(tr[:, 0], tr[:, 1], color="#888")
    ax.add_patch(robot_patch(tr[0], color="#bbb")); ax.add_patch(robot_patch(tr[-1]))
    ax.plot(gx, gy, "*", ms=16, color="#2E9E5B")
    ax.set_aspect("equal"); ax.grid(alpha=.3); ax.set_xlim(-4, 4); ax.set_ylim(-4, 4)
    ax.set_title(f"go-to-goal, {len(tr) * 0.05:.1f} s")
    plt.show()

interact(show_goal,
         gx=FloatSlider(3.0, min=-3.5, max=3.5, step=0.25),
         gy=FloatSlider(2.0, min=-3.5, max=3.5, step=0.25),
         start_heading_deg=FloatSlider(0, min=-180, max=180, step=15));

---
# Part C: a warehouse and A*

A planner does not work on the continuous floor but on a **grid** of 0.5 m cells, each free or blocked. We draw a 15 m × 10 m warehouse with three rows of shelves and then **inflate** the shelves by the robot's radius, so that the planner can treat the robot as a single point.

**A\*** searches the grid for the cheapest path. It always expands the cell with the lowest cost so far plus a straight-line guess of the distance still to go. What comes out is a list of **waypoints**, with no speeds or timing; that is the controller's job.

Other planners you will come across: Dijkstra (A\* without the guess), D\* Lite (fast replanning), Theta\* (paths not tied to the grid directions), Hybrid A\* (car-like robots), and the sampling-based PRM, RRT and RRT\*.

This part is complete: run the cell and look at the path.

In [ ]:
CELL = 0.5                      # metres per grid cell
W, H = 30, 20                   # grid size in cells: 15 m x 10 m

def make_warehouse():
    grid = np.zeros((H, W), dtype=int)          # 0 = free, 1 = blocked
    grid[0, :] = grid[-1, :] = grid[:, 0] = grid[:, -1] = 1
    for r in (4, 9, 14):                          # three shelf rows, two blocks each
        grid[r:r+2, 4:11]  = 1
        grid[r:r+2, 16:23] = 1
    return grid

def inflate(grid, radius_cells=1):
    """Grow every blocked cell by the robot's radius."""
    out = grid.copy()
    for r, c in np.argwhere(grid == 1):
        out[max(0, r-radius_cells):r+radius_cells+1, max(0, c-radius_cells):c+radius_cells+1] = 1
    return out

def to_cell(p):   return (int(round(p[1] / CELL)), int(round(p[0] / CELL)))   # (x, y) in metres -> (row, col)
def to_world(rc): return np.array([rc[1] * CELL, rc[0] * CELL])               # (row, col) -> (x, y) in metres

def astar(grid, start_rc, goal_rc):
    """A* on an 8-connected grid. Returns a list of (row, col) cells, or None if there is no path."""
    nbrs = [(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(-1,1),(1,-1),(1,1)]
    h = lambda a, b: np.hypot(a[0]-b[0], a[1]-b[1])          # the guess: straight-line distance
    open_set = [(h(start_rc, goal_rc), 0.0, start_rc)]
    came, g = {}, {start_rc: 0.0}
    while open_set:
        _, gc, cur = heapq.heappop(open_set)
        if cur == goal_rc:
            path = [cur]
            while cur in came: cur = came[cur]; path.append(cur)
            return path[::-1]
        for dr, dc in nbrs:
            n = (cur[0]+dr, cur[1]+dc)
            if not (0 <= n[0] < grid.shape[0] and 0 <= n[1] < grid.shape[1]): continue
            if grid[n] == 1: continue
            ng = gc + np.hypot(dr, dc)
            if ng < g.get(n, np.inf):
                g[n] = ng; came[n] = cur
                heapq.heappush(open_set, (ng + h(n, goal_rc), ng, n))
    return None

grid = make_warehouse()
cspace = inflate(grid, radius_cells=1)

def draw_map(ax, grid):
    ax.imshow(grid, cmap="Greys", origin="lower",
              extent=[-CELL/2, (W-0.5)*CELL, -CELL/2, (H-0.5)*CELL], alpha=0.8)
    ax.set_xlim(-CELL/2, (W-0.5)*CELL); ax.set_ylim(-CELL/2, (H-0.5)*CELL)
    ax.set_aspect("equal"); ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]")

start, goal = (1.0, 1.0), (13.0, 6.0)
path = astar(cspace, to_cell(start), to_cell(goal))
pts = np.array([to_world(c) for c in path])

fig, ax = plt.subplots(figsize=(8, 5.5))
draw_map(ax, grid)
ax.plot(pts[:, 0], pts[:, 1], "-o", ms=3, color="#378ADD", label="A* path (waypoints)")
ax.plot(*start, "s", ms=9, color="#D85A30", label="start"); ax.plot(*goal, "*", ms=16, color="#2E9E5B", label="goal")
ax.legend(loc="upper right"); ax.set_title(f"A*: {len(path)} waypoints")
plt.show()

The path keeps clear of the inflated shelves and cuts corners diagonally. If you move `goal` into a shelf, `astar` returns `None`: there is no path, and a real robot has to do something sensible with that answer. Parts D and E are about exactly that.

---
# Part D: a list of goals and a state machine

Now the robot gets several goals and visits them in order. We describe this behaviour as a **finite state machine**: the robot is always in exactly one state, and simple rules decide when it switches to the next. The code calls the current state `mode`, to keep it apart from the robot's position `state`.

```
IDLE --order--> PLAN --path--> MOVE --close enough--> ARRIVED --more goals--> PLAN
                                                              \--no more----> DONE
```

| State | What happens |
|---|---|
| `IDLE` | waits for an order, a list of goals |
| `PLAN` | runs A* from the robot to the next goal |
| `MOVE` | follows the waypoints with the go-to-goal controller from Part B |
| `ARRIVED` | ticks off the goal it has just reached |
| `DONE` | stops, nothing is left |

Almost all of the machine is written. You supply the two rules that end `MOVE` and `ARRIVED`.

### Exercise D1: when has the robot arrived?

In `MOVE` the robot keeps driving until it is close enough to the goal, which here means closer than the tolerance `goal_tol` = 0.15 m. At that moment the machine switches to `ARRIVED`. The state machine measures `dist_to_goal` for you.

Example: 0.05 m from the goal the robot has arrived; 2 m away it keeps driving.

A comparison such as `a < b` is itself a value, `True` or `False`, so its result can be stored in a variable directly, without an `if`.

Your line: `is_close` is `dist_to_goal < goal_tol`.

In [ ]:
def arrived(dist_to_goal, goal_tol=0.15):
    """True when the robot is close enough to the goal to leave MOVE."""
    # ---- your code: 1 line ----
    is_close = False
    # ----------------------------
    return is_close

In [ ]:
# Check
def _check_D1():
    near, far = bool(arrived(0.05)), bool(arrived(2.0))
    if near and not far:
        return "Correct: MOVE switches to ARRIVED as soon as the robot is closer than goal_tol."
    if not near and not far:
        return "Not yet: arrived(...) is always False, so the robot would never stop. Compare dist_to_goal with goal_tol using <."
    if near and far:
        return "Not yet: arrived(...) is True even 2 m from the goal, so the robot would stop far too early. Compare dist_to_goal with goal_tol."
    return "Not yet: it is the other way round. Close means dist_to_goal is smaller than goal_tol."

try:
    print(_check_D1())
except Exception as e:
    print(f"Not yet: the check could not run your code ({type(e).__name__}: {e}). Run the cell above and look for a typo.")

### Exercise D2: after arriving, what next?

In `ARRIVED` the goal that was just reached has already been removed from the list. If goals are left, the robot goes back to `PLAN` to plan the way to the next one; if the list is empty, the mission is `DONE`.

Example: `goals_left = [(13.0, 6.0), (1.0, 1.0)]` gives `"PLAN"`, and `goals_left = []` gives `"DONE"`.

`len(goals_left)` is the number of items in the list, so `len(goals_left) > 0` is `True` when something is left. The expression `a if condition else b` picks `a` when the condition is `True` and `b` when it is `False`, all in one line.

Your line: `next_mode` is `"PLAN"` if `len(goals_left) > 0`, else `"DONE"`.

In [ ]:
def after_arrived(goals_left):
    """The state that follows ARRIVED: "PLAN" if goals are left, otherwise "DONE"."""
    # ---- your code: 1 line ----
    next_mode = "DONE"
    # ----------------------------
    return next_mode

In [ ]:
# Check
def _check_D2():
    more, empty = after_arrived([(13.0, 6.0), (1.0, 1.0)]), after_arrived([])
    if more == "PLAN" and empty == "DONE":
        return "Correct: more goals lead back to PLAN, an empty list ends in DONE."
    if more == "DONE" and empty == "DONE":
        return 'Not yet: after_arrived always returns "DONE", so the robot would stop after the first goal. With goals left it must return "PLAN".'
    if more == "PLAN" and empty == "PLAN":
        return 'Not yet: after_arrived always returns "PLAN", so the mission would never end. With an empty list it must return "DONE".'
    return f'Not yet: expected "PLAN" and "DONE", but got {more!r} and {empty!r}. Check the spelling, in capital letters.'

try:
    print(_check_D2())
except Exception as e:
    print(f"Not yet: the check could not run your code ({type(e).__name__}: {e}). Run the cell above and look for a typo.")

### Running the mission

The next cell runs the whole state machine with your four pieces: `step` and `go_to_goal` inside `MOVE`, `arrived` to leave `MOVE`, and `after_arrived` to leave `ARRIVED`. The robot visits three pick points and then drives back to the start.

In [ ]:
def run_mission(goals, start=(1.0, 1.0, 0.0), grid=cspace, dt=0.05, T_max=120.0, wp_tol=0.25, goal_tol=0.15):
    """Run the mission state machine. Returns the trajectory, the state at every step, the planned paths and the reached goals."""
    state = np.array(start, float)
    mode, goals = "IDLE", list(goals)
    path, paths, traj, modes, reached = [], [], [], [], []
    for _ in range(int(T_max / dt)):
        traj.append(state.copy()); modes.append(mode)
        if mode == "IDLE":
            mode = "PLAN" if goals else "DONE"
        elif mode == "PLAN":
            cells = astar(grid, to_cell(state[:2]), to_cell(goals[0]))
            if cells is None:
                print("no path to", goals[0], "-> skipping it"); goals.pop(0); mode = "IDLE"
            else:
                path = [to_world(c) for c in cells][1:]; paths.append(np.array(path)); mode = "MOVE"
        elif mode == "MOVE":
            d = lambda p: np.hypot(*(p - state[:2]))
            while len(path) > 1 and (d(path[0]) < wp_tol or d(path[1]) < d(path[0])):
                path.pop(0)                                  # passed a waypoint: drop it
            target = path[min(2, len(path)-1)] if len(path) > 1 else goals[0]   # aim about 1 m ahead
            dist_to_goal = np.hypot(*(np.array(goals[0]) - state[:2]))
            if arrived(dist_to_goal, goal_tol):              # exercise D1
                mode = "ARRIVED"; continue
            v, w = go_to_goal(state, target)                 # exercise B1
            state = step(state, v, w, dt)                    # exercise A1
        elif mode == "ARRIVED":
            reached.append(goals.pop(0))
            mode = after_arrived(goals)                      # exercise D2
        elif mode == "DONE":
            break
    return np.array(traj), modes, paths, reached

mission = [(6.5, 4.0), (13.0, 6.0), (1.0, 8.5), (1.0, 1.0)]      # three pick points, then home
traj, modes, paths, reached = run_mission(mission)
t_end = len(traj) * 0.05
if modes[-1] == "DONE" and len(reached) == len(mission):
    print(f"Mission complete: all {len(mission)} goals reached in {t_end:.1f} s, states visited: {', '.join(sorted(set(modes)))}.")
else:
    print(f"Mission not complete after {t_end:.1f} s: {len(reached)} of {len(mission)} goals reached, last state {modes[-1]}. "
          "The four checks above should all say Correct.")

### Watching it drive

The animation takes a few seconds to prepare.

In [ ]:
def animate_mission(traj, modes, paths, goals, max_frames=200, interval=40):
    fig, ax = plt.subplots(figsize=(8, 5.5))
    draw_map(ax, grid)
    for p in paths:
        if len(p): ax.plot(p[:, 0], p[:, 1], "--", lw=1, color="#378ADD", alpha=.6)
    for k, g in enumerate(goals):
        ax.plot(*g, "*", ms=14, color="#2E9E5B"); ax.annotate(str(k+1), g, xytext=(4, 4), textcoords="offset points")
    trail, = ax.plot([], [], color="#D85A30", lw=1.5)
    body = ax.add_patch(robot_patch(traj[0]))
    label = ax.text(0.02, 0.96, "", transform=ax.transAxes, va="top", fontsize=11,
                    bbox=dict(boxstyle="round", fc="white", alpha=.8))
    every = max(1, len(traj) // max_frames)
    idx = list(range(0, len(traj), every)) + [len(traj) - 1]
    def update(i):
        k = idx[i]
        trail.set_data(traj[:k+1, 0], traj[:k+1, 1])
        body.set_xy(robot_patch(traj[k]).get_xy())
        label.set_text(f"t = {k * 0.05:5.1f} s   state: {modes[k]}")
        return trail, body, label
    anim = animation.FuncAnimation(fig, update, frames=len(idx), interval=interval, blit=True)
    plt.close(fig)
    return anim

anim = animate_mission(traj, modes, paths, mission)
HTML(anim.to_jshtml())

A GIF plays anywhere, including in your H1 report. After running the next cell, download `mission.gif` from the Files panel (the folder icon on the left).

In [ ]:
anim.save("mission.gif", writer="pillow", fps=20)
print("saved mission.gif: download it from the Files panel on the left")

The state over time: every step in the line is a switch, so you can read off how long the robot spent planning, driving and arriving.

In [ ]:
order = ["IDLE", "PLAN", "MOVE", "ARRIVED", "DONE"]
fig, ax = plt.subplots(figsize=(8, 2.4))
ax.step(np.arange(len(modes)) * 0.05, [order.index(m) for m in modes], where="post", color="#D85A30")
ax.set_yticks(range(len(order))); ax.set_yticklabels(order); ax.set_xlabel("time [s]"); ax.grid(alpha=.3)
ax.set_title("State machine: the state over time")
plt.show()

---
# Part E: a blocked aisle and a first behaviour tree

Suppose an aisle turns out to be blocked while the robot is driving. The state machine needs a new exit from `MOVE`, `PLAN` has to know it is re-planning, and a battery check later would add another exit to every state. State machines collect arrows quickly.

A **behaviour tree** arranges the same logic as a tree of small tasks. The tree is **ticked** from the root many times a second, and each task answers `SUCCESS`, `FAILURE` or `RUNNING` (still busy).

| Node | Rule |
|---|---|
| **Sequence** | runs its children in order and stops at the first one that does not return `SUCCESS`: "do this, then that" |
| **Fallback** (`Selector` in py_trees) | tries its children in order until one does not return `FAILURE`: "try this, otherwise that" |

Recovery is then a single Fallback: follow the path, and if that fails, re-plan. The tasks below only pretend to drive, so that the shape of the tree is easy to see. Nav2, the ROS 2 navigation stack, is built the same way.

Nothing to write here: run the next two cells and read the printout. The first one installs the small py_trees library, which takes about ten seconds the first time.

In [ ]:
try:
    import py_trees
except ImportError:                       # Colab: install it once
    import subprocess, sys
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "py_trees"])
    if r.returncode:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--break-system-packages", "py_trees"], check=True)
    import py_trees
from py_trees.common import Status

class HasGoal(py_trees.behaviour.Behaviour):
    def __init__(self, bb): super().__init__("has goal?"); self.bb = bb
    def update(self): return Status.SUCCESS if self.bb["goals"] else Status.FAILURE

class FollowPath(py_trees.behaviour.Behaviour):
    def __init__(self, bb): super().__init__("follow path"); self.bb = bb
    def update(self):
        if self.bb["blocked"]:           # the world changed under us
            return Status.FAILURE
        self.bb["progress"] += 1
        return Status.SUCCESS if self.bb["progress"] >= 3 else Status.RUNNING

class Replan(py_trees.behaviour.Behaviour):
    def __init__(self, bb): super().__init__("re-plan"); self.bb = bb
    def update(self):
        self.bb["blocked"] = False; self.bb["progress"] = 0
        return Status.RUNNING            # recovered, but the goal is not reached yet: keep going

class PopGoal(py_trees.behaviour.Behaviour):
    def __init__(self, bb): super().__init__("goal reached, next"); self.bb = bb
    def update(self): self.bb["goals"].pop(0); self.bb["progress"] = 0; return Status.SUCCESS

bb = {"goals": ["shelf A", "shelf B"], "blocked": False, "progress": 0}

drive = py_trees.composites.Selector("drive or recover", memory=False,        # the Fallback
            children=[FollowPath(bb), Replan(bb)])
mission_tree = py_trees.composites.Sequence("mission", memory=True,
            children=[HasGoal(bb), drive, PopGoal(bb)])

print(py_trees.display.unicode_tree(mission_tree))

In [ ]:
# Tick the tree; on tick 2 the aisle becomes blocked.
for t in range(1, 9):
    if t == 2:
        bb["blocked"] = True
    mission_tree.tick_once()
    print(f"tick {t}: root -> {mission_tree.status.name:8s}  drive -> {drive.status.name:8s}  goals left: {bb['goals']}")
    if not bb["goals"]:
        break
print("Behaviour tree finished: every goal reached despite the blocked aisle." if not bb["goals"]
      else "The tree is still running: tick it a few more times.")

On tick 2 `follow path` returns `FAILURE`, because the aisle is blocked, so the Fallback runs `re-plan`. That returns `RUNNING`: recovered, but not at the goal yet. The following ticks finish the goal. Adding the recovery did not change any of the existing tasks, and that is the main argument for behaviour trees.

---
## Summary

* A differential-drive robot cannot move sideways (**nonholonomic**); its paths are pieced together from arcs and straight lines.
* Go-to-goal needs two errors, the **distance error** ρ and the **heading error** α wrapped to ±180°, and two proportional rules.
* The **planner** decides which way, the **controller** how fast. Keeping the two apart keeps the classical stack manageable.
* A **finite state machine** describes behaviour as a sequence of states; a **behaviour tree** keeps it organised once recovery is needed.

The animation from Part D is the kind of result you hand in with homework H1.